# Evaluation

Notebook to launch `evaluation/evaluate.py` on generated images and save quantitative results.

In [ ]:
import sys
import torch

print(f'Python:  {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')

In [ ]:
!pip install -q \
    torchmetrics==1.3.2 \
    torch-fidelity==0.3.0 \
    clean-fid==0.1.35 \
    lpips==0.1.4 \
    Pillow pyyaml numpy pandas scipy

In [ ]:
from pathlib import Path
import os

REPO_NAME = 'Few-Shot-Personalization-of-a-Diffusion-Model-for-Industrial-Defect-Synthesis'
REPO_URL = f'https://github.com/alecanc/{REPO_NAME}'
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
CLONE_TARGET = WORK / REPO_NAME

def is_repo(path):
    return (path / 'evaluation' / 'evaluate.py').exists()

candidates = [Path.cwd(), *Path.cwd().parents, CLONE_TARGET, WORK / 'defect-synthesis']
repo = next((p for p in candidates if is_repo(p)), None)

if repo is None:
    if CLONE_TARGET.exists():
        raise FileNotFoundError(f'{CLONE_TARGET} exists but evaluation/evaluate.py was not found inside it')
    !git clone {REPO_URL} {CLONE_TARGET}
    repo = CLONE_TARGET

if not is_repo(repo):
    raise FileNotFoundError(f'Repository not found correctly: {repo}')

os.chdir(repo)
CONFIG = repo / 'defect-synthesis' / 'config.yaml'
if not CONFIG.exists():
    raise FileNotFoundError(f'Config not found: {CONFIG}')
RESULTS = Path('/kaggle/working/results') if Path('/kaggle').exists() else repo / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

print(f'Repository: {repo}')
print(f'Config:     {CONFIG}')
print(f'Results:    {RESULTS}')

In [ ]:
import sys
import subprocess
import pandas as pd
from pathlib import Path

WORK = Path('/kaggle/working')
repo_name = 'Few-Shot-Personalization-of-a-Diffusion-Model-for-Industrial-Defect-Synthesis'
repo = WORK / repo_name
evaluate_path = repo / 'evaluation' / 'evaluate.py'

final_out_dir = WORK / "eval_result_final"
final_out_dir.mkdir(parents=True, exist_ok=True)
visuals_dest = final_out_dir / "visuals"
visuals_dest.mkdir(parents=True, exist_ok=True)

CONFIG = repo / 'defect-synthesis' / 'config.yaml'
temp_csvs = []

evaluation_jobs = [
    {"conditions": ["zero_shot", "stage1_only", "singlestage", "singlestage_full", "sweep_prompt", "sweep_weight"], "category": "bottle", "defect_type": None},
    {"conditions": ["zero_shot", "stage1_only", "singlestage", "singlestage_full", "sweep_prompt", "sweep_weight"], "category": "metal_nut", "defect_type": None},
    {"conditions": ["zero_shot", "stage1_only", "singlestage", "singlestage_full", "sweep_prompt", "sweep_weight"], "category": "leather", "defect_type": None},
]

print("Starting final evaluation...")

for idx, job in enumerate(evaluation_jobs):
    temp_output = final_out_dir / f"temp_eval_{idx}.csv"
    temp_csvs.append(temp_output)
    
    cmd = [
        sys.executable, str(evaluate_path),
        '--config', str(CONFIG),
        '--conditions', *job["conditions"],
        '--metrics', 'fid', 'kid', 'lpips', 'dino',
        '--device', 'cuda',
        '--output', str(temp_output),
        '--category', job["category"],
        '--visualize',
        '--visuals_dir', str(visuals_dest)
    ]
    
    if job["defect_type"]:
        cmd += ['--defect_type', job["defect_type"]]
        
    print(f"   evaluating: {job['category']} | Defect: {job['defect_type'] or 'All'} | Conditions: {job['conditions']}")
    subprocess.run(cmd, check=True)


print("Final results compilation...")
dfs = []
for f in temp_csvs:
    if f.exists():
        dfs.append(pd.read_csv(f))
        f.unlink()

if dfs:
    merged_df = pd.concat(dfs, ignore_index=True)
    merged_df.to_csv(final_out_dir / "eval_summary_complete.csv", index=False)
    print(f"Final results saved in: {final_out_dir}/eval_summary_complete.csv")
    print(merged_df.to_markdown(index=False))
else:
    print("No results generated.")

In [ ]:
import os
import re
import json
import shutil
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

print("generating dynamic grids for visual comparison")
summary_dest = Path("/kaggle/working/eval_result_final/summary_grids")
summary_dest.mkdir(parents=True, exist_ok=True)
generated_root = Path("/kaggle/working/generated")

def crop_first_image_from_grid(im):
    w, h = im.size
    if w == h:
        return im
    
    ratio = w / h
    if ratio >= 3.0:
        size = h
    elif ratio >= 1.4 and ratio <= 2.2:
        size = h // 2
    else:
        size = h
    return im.crop((0, 0, size, size))

def filter_images_by_step(images_list, target_step=400):
    step_pattern = re.compile(r'step(\d+)_img')
    
    step_groups = {}
    for img in images_list:
        match = step_pattern.search(img.name)
        if match:
            step_num = int(match.group(1))
            if step_num not in step_groups:
                step_groups[step_num] = []
            step_groups[step_num].append(img)
            
    if not step_groups:
        return images_list 
        
    available_steps = sorted(step_groups.keys())
    selected_step = None
    for step in reversed(available_steps):
        if step <= target_step:
            selected_step = step
            break
            
    if selected_step is None:
        selected_step = available_steps[0]  
        
    return sorted(step_groups[selected_step])

def find_images_for_method_recursively(category, defect, method_key, max_imgs=6):
    if not generated_root.exists():
        return []

    cat_norm = category.lower().replace("_", "").replace("-", "")
    def_norm = defect.lower().replace("_", "").replace("-", "")
    method_norm = method_key.lower().replace("-", "").replace("_", "").replace(" ", "").replace("(", "").replace(")","")

    matched_dirs = []
    for root, dirs, files in os.walk(str(generated_root)):
        for d in dirs:
            dir_path = Path(root) / d
            dir_path_str = str(dir_path).lower().replace("_", "").replace("-", "")
            
            if cat_norm in dir_path_str and def_norm in dir_path_str:
                is_match = False
                if method_norm == "zeroshot":
                    if "zero" in dir_path_str and "shot" in dir_path_str:
                        is_match = True
                elif method_norm == "singlestage":
                    if "single" in dir_path_str and "stage" in dir_path_str and "full" not in dir_path_str:
                        is_match = True
                elif method_norm == "singlestagefull":
                    if "single" in dir_path_str and "stage" in dir_path_str and "full" in dir_path_str:
                        is_match = True
                elif method_norm in ("twostageweight", "sweepweight"):
                    if "weight" in dir_path_str and "sweep" in dir_path_str:
                        is_match = True
                elif method_norm in ("twostageprompt", "sweepprompt"):
                    if "prompt" in dir_path_str and "sweep" in dir_path_str:
                        is_match = True
                
                if is_match:
                    matched_dirs.append(dir_path)

    matched_dirs = sorted(matched_dirs, key=lambda p: (1 if "validation" in str(p).lower() else 0, len(str(p))), reverse=True)

    for path in matched_dirs:
        all_images = sorted(list(path.glob("*.png")) + list(path.glob("*.jpg")) + list(path.glob("*.jpeg")))
        if all_images:
            if "singlestage" in method_norm:
                all_images = filter_images_by_step(all_images, target_step=400)
            return all_images[:max_imgs]

    return []

categories_defects = {
    "bottle": ["broken_large", "broken_small", "contamination"],
    "metal_nut": ["bent", "color", "flip", "scratch"],
    "leather": ["color", "cut", "fold", "glue", "poke"]
}

methods_order = [
    "Real", "Zero-Shot", "Single-Stage",
    "Single-Stage Full", "Two-Stage (Weight)", "Two-Stage (Prompt)"
]

for category, defects in categories_defects.items():
    split_path = Path("/kaggle/working/splits") / f"{category}_split.json"
    split_data = {}
    if split_path.exists():
        with open(split_path, "r") as sf:
            split_data = json.load(sf)

    for defect in defects:
        real_paths_all = split_data.get("eval", {}).get(defect, [])
        real_paths = [Path(p) for p in real_paths_all if Path(p).exists()][:6]

        active_methods = {}
        
        if real_paths:
            loaded_reals = []
            for rp in real_paths:
                try:
                    loaded_reals.append(Image.open(rp).convert("RGB").resize((256, 256)))
                except Exception:
                    pass
            if loaded_reals:
                active_methods["Real"] = loaded_reals

        for label in [m for m in methods_order if m != "Real"]:
            imgs_paths = find_images_for_method_recursively(category, defect, label, max_imgs=6)
            print(f"      [{category}/{defect}] Method: {label} | Found paths: {len(imgs_paths)}")
            if imgs_paths:
                loaded_imgs = []
                for p in imgs_paths:
                    try:
                        im = Image.open(p).convert("RGB")
                        im = crop_first_image_from_grid(im)
                        im = im.resize((256, 256))
                        loaded_imgs.append(im)
                    except Exception as e:
                        print(f" Error loading {p.name}: {e}")
                        pass
                if loaded_imgs:
                    active_methods[label] = loaded_imgs

        if not active_methods:
            continue

        img_size = 256
        padding = 12
        header_height = 55
        label_area_width = 180
        n_cols = 6  
        
        grid_width = label_area_width + n_cols * img_size + (n_cols + 2) * padding
        grid_height = header_height + len(active_methods) * (img_size + padding) + padding

        grid = Image.new("RGB", (grid_width, grid_height), color=(30, 30, 36)) 
        draw = ImageDraw.Draw(grid)
        
        try:
            font = ImageFont.load_default()
        except Exception:
            font = None

        title = f"{category.upper()} - {defect.upper().replace('_', ' ')}"
        draw.text((padding + 5, 18), title, fill=(255, 215, 0), font=font)
        
        for row_idx, (method_name, images) in enumerate(active_methods.items()):
            y = header_height + padding + row_idx * (img_size + padding)
            draw.text((padding + 10, y + img_size // 2 - 10), method_name, fill=(255, 255, 255), font=font)
            
            for col_idx, img in enumerate(images):
                x = label_area_width + padding + col_idx * (img_size + padding)
                grid.paste(img, (x, y))

        grid.save(summary_dest / f"{category}_{defect}_summary_comparison.png")
        print(f" Summary grid {category}/{defect} ({len(active_methods)} models)")

In [ ]:
from IPython.display import FileLink, display
import shutil
import os
from pathlib import Path

print("Creating zip file")


repo_visuals = repo / "results" / "visuals"
if repo_visuals.exists():
    for f in repo_visuals.glob("*.png"):
        shutil.copy(f, visuals_dest)

shutil.make_archive(str(WORK / "eval_result_final"), 'zip', str(final_out_dir))
print(f"zip created: {WORK}/eval_result_final.zip")

zip_path = WORK / "eval_result_final.zip"
rel_path = os.path.relpath(zip_path, os.getcwd())

print(f"Link to download :")
display(FileLink(rel_path))